# Lab 1: AI Security Threat Modeling

## Learning Objectives
By the end of this lab, you will:
- Define and structure the assets of an LLM application using Python dataclasses
- Build a Data Flow Diagram (DFD) with `networkx` and visualize trust boundaries
- Apply the STRIDE methodology to each data flow and produce a threat matrix
- Score threats by risk (Impact × Likelihood) and generate a priority heat map

## Setup
Run the cell below to install required libraries.

In [ ]:
!uv pip install networkx matplotlib pandas seaborn -q

---
## Part 1: Define Application Assets

A threat model starts with knowing **what you're protecting**. Assets fall into three categories: data, model, and system assets.

In [ ]:
from dataclasses import dataclass, field
from typing import List
from enum import Enum


class AssetCategory(Enum):
    DATA = "data"
    MODEL = "model"
    SYSTEM = "system"


class Sensitivity(Enum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3
    CRITICAL = 4


@dataclass
class Asset:
    name: str
    category: AssetCategory
    sensitivity: Sensitivity
    description: str
    location: str  # Where it lives (e.g., "vector_db", "env_var", "memory")


# Predefined assets for the Research Assistant
RESEARCH_ASSISTANT_ASSETS = [
    Asset("API Keys",         AssetCategory.DATA,   Sensitivity.CRITICAL, "Anthropic/OpenAI API keys", "env_var"),
    Asset("System Prompt",    AssetCategory.DATA,   Sensitivity.HIGH,     "Core instructions defining assistant behavior", "memory"),
    Asset("User Query History",AssetCategory.DATA,  Sensitivity.HIGH,     "All user conversations", "database"),
    Asset("Vector Database",  AssetCategory.DATA,   Sensitivity.HIGH,     "Proprietary research embeddings", "vector_db"),
    Asset("Embedding Model",  AssetCategory.MODEL,  Sensitivity.MEDIUM,   "Text-to-vector model", "local"),
    Asset("Prompt Templates", AssetCategory.MODEL,  Sensitivity.MEDIUM,   "Curated prompt engineering templates", "file"),
    Asset("Backend Server",   AssetCategory.SYSTEM, Sensitivity.HIGH,     "Application server + runtime", "cloud"),
    Asset("External APIs",    AssetCategory.SYSTEM, Sensitivity.MEDIUM,   "Web search, financial data APIs", "external"),
]

print(f"Total assets: {len(RESEARCH_ASSISTANT_ASSETS)}")
print()
print(f"{'Asset':<25} {'Category':<10} {'Sensitivity':<10} {'Location'}")
print("-" * 70)
for asset in sorted(RESEARCH_ASSISTANT_ASSETS, key=lambda a: a.sensitivity.value, reverse=True):
    print(f"{asset.name:<25} {asset.category.value:<10} {asset.sensitivity.name:<10} {asset.location}")

### Exercise 1.1: Add Two More Assets

Think about what else the Research Assistant uses or produces. Add **two additional assets** to the list below.

In [ ]:
# TODO: Add two more assets relevant to your Research Assistant
my_assets = [
    Asset(
        name="",           # e.g., "Generated Reports"
        category=AssetCategory.DATA,
        sensitivity=Sensitivity.MEDIUM,
        description="",    # Describe what this asset is
        location="",       # Where does it live?
    ),
    Asset(
        name="",
        category=AssetCategory.SYSTEM,
        sensitivity=Sensitivity.HIGH,
        description="",
        location="",
    ),
]

all_assets = RESEARCH_ASSISTANT_ASSETS + [a for a in my_assets if a.name]
print(f"Total assets (including yours): {len(all_assets)}")

---
## Part 2: Build a Data Flow Diagram

A DFD shows how data moves through your system and identifies **trust boundaries** — the lines attackers try to cross.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Define nodes with their trust level
nodes = {
    # External (untrusted) - red
    "User": {"trust": "untrusted", "type": "external"},
    "Web Search": {"trust": "untrusted", "type": "external"},
    "Uploaded Docs": {"trust": "untrusted", "type": "external"},

    # Internal processing - blue
    "Input Validator": {"trust": "trusted", "type": "process"},
    "LLM Engine": {"trust": "trusted", "type": "process"},
    "Output Guard": {"trust": "trusted", "type": "process"},

    # Storage - green
    "Vector DB": {"trust": "trusted", "type": "storage"},
    "Audit Log": {"trust": "trusted", "type": "storage"},
}

# Define data flows (edges) with labels
flows = [
    ("User",           "Input Validator",  "Raw query"),
    ("Input Validator", "LLM Engine",      "Sanitized + hardened prompt"),
    ("Web Search",     "LLM Engine",       "Retrieved web content (untrusted)"),
    ("Uploaded Docs",  "Input Validator",  "Document content (untrusted)"),
    ("Vector DB",      "LLM Engine",       "Relevant context chunks"),
    ("LLM Engine",     "Output Guard",     "Raw LLM response"),
    ("Output Guard",   "User",             "Safe response"),
    ("Input Validator","Audit Log",        "Security events"),
    ("Output Guard",   "Audit Log",        "Output violations"),
]

G = nx.DiGraph()
for name, attrs in nodes.items():
    G.add_node(name, **attrs)
for src, dst, label in flows:
    G.add_edge(src, dst, label=label)

# Layout and colors
pos = {
    "User":            (0, 1),
    "Uploaded Docs":   (0, 0),
    "Web Search":      (0, -1),
    "Input Validator": (2, 0.5),
    "LLM Engine":      (4, 0),
    "Output Guard":    (6, 0),
    "Vector DB":       (3, -1.5),
    "Audit Log":       (4, 1.5),
}

color_map = {
    "untrusted": "#FF7A5C",
    "trusted":   "#00C9A7",
}
node_colors = [color_map[nodes[n]["trust"]] for n in G.nodes()]

fig, ax = plt.subplots(figsize=(14, 8))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2500, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold", ax=ax)
nx.draw_networkx_edges(G, pos, edge_color="#1C355E", arrows=True,
                       arrowsize=20, width=2, ax=ax,
                       connectionstyle="arc3,rad=0.1")
edge_labels = {(s, d): l for s, d, l in flows}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax)

# Legend
legend = [
    mpatches.Patch(color="#FF7A5C", label="Untrusted (attack surface)"),
    mpatches.Patch(color="#00C9A7", label="Trusted (internal)"),
]
ax.legend(handles=legend, loc="upper right", fontsize=10)
ax.set_title("Research Assistant — Data Flow Diagram", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Nodes: {G.number_of_nodes()} | Edges (data flows): {G.number_of_edges()}")

### Exercise 2.1: Identify Trust Boundary Crossings

A **trust boundary crossing** is an edge from an untrusted node to a trusted node — these are your highest-risk attack surfaces.

In [ ]:
# TODO: Find all edges that cross a trust boundary (untrusted → trusted)
trust_boundary_crossings = []

for src, dst, data in G.edges(data=True):
    src_trust = G.nodes[src]["trust"]
    dst_trust = G.nodes[dst]["trust"]
    # TODO: Check if this edge crosses a trust boundary
    # A crossing happens when src_trust == "untrusted" and dst_trust == "trusted"
    pass  # Replace with your logic

print("Trust boundary crossings (highest attack risk):")
for crossing in trust_boundary_crossings:
    print(f"  {crossing}")

# Validation
assert len(trust_boundary_crossings) >= 2, "Should find at least 2 trust boundary crossings"
print("\n✅ Found trust boundary crossings!")

---
## Part 3: Apply STRIDE — Build the Threat Matrix

STRIDE maps threat categories to each data flow. For each trust boundary crossing, identify potential threats in each STRIDE category.

In [ ]:
import pandas as pd

# STRIDE threat matrix for the Research Assistant
threat_data = [
    # Data flow, STRIDE category, Threat description, Impact, Likelihood
    ("User → Input Validator",     "Spoofing",     "Attacker impersonates legitimate user with stolen session token", 8, 4),
    ("User → Input Validator",     "Tampering",    "Prompt injection to modify system instructions", 9, 9),
    ("User → Input Validator",     "Repudiation",  "User denies submitting malicious query (no audit trail)", 5, 6),
    ("User → Input Validator",     "Info Disc.",   "User probes for system prompt / internal state", 8, 8),
    ("User → Input Validator",     "DoS",          "Expensive recursive queries exhaust API budget", 6, 8),
    ("User → Input Validator",     "Elevation",    "Injection causes LLM to execute restricted tools", 9, 7),

    ("Web Search → LLM Engine",    "Spoofing",     "Attacker poisons search results with fake sources", 7, 5),
    ("Web Search → LLM Engine",    "Tampering",    "Indirect prompt injection in retrieved web content", 8, 7),
    ("Web Search → LLM Engine",    "Info Disc.",   "Retrieved content leaks sensitive info to attacker", 6, 4),
    ("Web Search → LLM Engine",    "DoS",          "Retrieved content triggers expensive processing loop", 5, 3),

    ("Uploaded Docs → Input Validator", "Tampering",  "Malicious doc with embedded injection payload", 8, 6),
    ("Uploaded Docs → Input Validator", "Info Disc.", "Document contains embedded exfiltration attack", 7, 5),

    ("LLM Engine → Output Guard",  "Info Disc.",   "LLM reveals system prompt or API key in response", 9, 3),
    ("LLM Engine → Output Guard",  "Tampering",    "LLM generates code that compromises downstream system", 8, 4),
]

columns = ["Data Flow", "STRIDE Category", "Threat Description", "Impact (1-10)", "Likelihood (1-10)"]
df = pd.DataFrame(threat_data, columns=columns)
df["Risk Score"] = df["Impact (1-10)"] * df["Likelihood (1-10)"]
df["Priority"] = pd.cut(
    df["Risk Score"],
    bins=[0, 20, 40, 60, 100],
    labels=["P3 — Low", "P2 — Medium", "P1 — High", "P0 — Critical"]
)

print(f"Total threats identified: {len(df)}")
print()
print(df.sort_values("Risk Score", ascending=False)[["STRIDE Category", "Threat Description", "Risk Score", "Priority"]].to_string(index=False))

### Exercise 3.1: Add STRIDE Threats for Vector DB

The Vector DB is a high-value asset. Fill in threats for the `Vector DB → LLM Engine` data flow.

In [ ]:
# TODO: Fill in at least 3 threats for the Vector DB → LLM Engine data flow
vectordb_threats = [
    # (data_flow, stride_category, description, impact, likelihood)
    ("Vector DB → LLM Engine", "Tampering",  "", 0, 0),   # What if someone writes poisoned vectors?
    ("Vector DB → LLM Engine", "Info Disc.", "", 0, 0),   # What private data could be retrieved?
    ("Vector DB → LLM Engine", "",           "", 0, 0),   # Add a third STRIDE category
]

df_new = pd.DataFrame(
    [(t[0], t[1], t[2], t[3], t[4]) for t in vectordb_threats if t[2]],
    columns=columns
)
if not df_new.empty:
    df_new["Risk Score"] = df_new["Impact (1-10)"] * df_new["Likelihood (1-10)"]
    df_full = pd.concat([df, df_new], ignore_index=True)
    print(f"Total threats after your additions: {len(df_full)}")
    print(df_new[["STRIDE Category", "Threat Description", "Risk Score"]].to_string(index=False))
else:
    df_full = df
    print("⚠️  Add threat descriptions above to complete the exercise.")

---
## Part 4: Risk Scoring & Heat Map

Risk = Impact × Likelihood. Visualize the priority landscape to guide mitigation effort.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Build risk matrix: rows = Impact bins, cols = Likelihood bins
df["Impact Bin"]     = pd.cut(df["Impact (1-10)"],     bins=[0,3,6,8,10], labels=["Low","Med","High","Crit"])
df["Likelihood Bin"] = pd.cut(df["Likelihood (1-10)"], bins=[0,3,6,8,10], labels=["Rare","Possible","Likely","Almost Certain"])

matrix = df.pivot_table(
    index="Impact Bin",
    columns="Likelihood Bin",
    values="Risk Score",
    aggfunc="count",
    fill_value=0,
).reindex(index=["Crit","High","Med","Low"],
          columns=["Rare","Possible","Likely","Almost Certain"],
          fill_value=0)

fig, ax = plt.subplots(figsize=(10, 6))
cmap = sns.color_palette(["#d4edda", "#fff3cd", "#f8d7da", "#721c24"], as_cmap=True)
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="YlOrRd",
    linewidths=0.5, ax=ax, cbar_kws={"label": "Number of Threats"}
)
ax.set_title("Research Assistant — Threat Risk Heat Map\n(cell value = number of threats)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Likelihood", fontsize=11)
ax.set_ylabel("Impact", fontsize=11)
plt.tight_layout()
plt.show()

# Top 5 threats
print("\nTop 5 Highest-Risk Threats:")
top5 = df.nlargest(5, "Risk Score")[["Data Flow","STRIDE Category","Threat Description","Risk Score","Priority"]]
print(top5.to_string(index=False))

### Exercise 4.1: Calculate Residual Risk

After implementing a security control, the **residual risk** drops. Estimate how much the `SecurityLayer` from Lesson 2 reduces the top threats.

In [ ]:
# TODO: For each top threat, estimate the residual likelihood after implementing SecurityLayer
# SecurityLayer includes: input sanitization + injection detection + prompt hardening

residual_estimates = {
    "Prompt injection to modify system instructions": {"original": 9, "residual": None},  # Fill: what likelihood remains?
    "User probes for system prompt / internal state": {"original": 8, "residual": None},
    "Injection causes LLM to execute restricted tools":{"original": 7, "residual": None},
    "Expensive recursive queries exhaust API budget":  {"original": 8, "residual": None},
    "Indirect prompt injection in retrieved web content":{"original": 7, "residual": None},
}

print(f"{'Threat':<50} {'Orig Likelihood':>15} {'Residual':>10} {'Reduction':>10}")
print("-" * 90)
for threat, vals in residual_estimates.items():
    orig = vals["original"]
    res  = vals["residual"]
    if res is not None:
        reduction = f"{(orig - res) / orig:.0%}"
        print(f"{threat:<50} {orig:>15} {res:>10} {reduction:>10}")
    else:
        print(f"{threat:<50} {orig:>15} {'TODO':>10} {'?':>10}")

---
## Reflection Questions

Answer these in the markdown cell below:

1. **Highest risk:** Which data flow has the most P0 threats? Why does it make sense?
2. **Defense priority:** Given the heat map, where should you invest security effort first?
3. **Residual risk:** After implementing all 4 security layers, what is the one threat you're most worried about? Why?

*Your answers here:*

1. ...
2. ...
3. ...

---
## Bonus: Export Threat Matrix as JSON Report

Generate a structured JSON security report that could be shared with stakeholders or ingested by a SIEM tool.

In [ ]:
import json
from datetime import datetime

def generate_threat_report(df: pd.DataFrame, app_name: str) -> dict:
    """Export threat matrix as structured JSON report."""
    threats = []
    for _, row in df.iterrows():
        threats.append({
            "id": f"T{len(threats)+1:03d}",
            "data_flow": row["Data Flow"],
            "stride_category": row["STRIDE Category"],
            "description": row["Threat Description"],
            "impact": int(row["Impact (1-10)"]),
            "likelihood": int(row["Likelihood (1-10)"]),
            "risk_score": int(row["Risk Score"]),
            "priority": str(row["Priority"]),
        })

    report = {
        "application": app_name,
        "generated_at": datetime.now().isoformat(),
        "total_threats": len(threats),
        "critical_count": sum(1 for t in threats if "Critical" in t["priority"]),
        "high_count":     sum(1 for t in threats if "High" in t["priority"]),
        "threats": sorted(threats, key=lambda t: t["risk_score"], reverse=True),
    }
    return report


report = generate_threat_report(df, "Gen-AI Research Assistant")

# Save to file
with open("threat_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(f"Report generated: {report['total_threats']} threats")
print(f"  Critical: {report['critical_count']}")
print(f"  High:     {report['high_count']}")
print(f"  Saved to: threat_report.json")
print()
print("Top 3 threats from report:")
for t in report["threats"][:3]:
    print(f"  [{t['id']}] {t['priority']} | {t['description'][:60]}... (Risk: {t['risk_score']})")